# TPWRS PRP Publications by Year

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.append(str(repo_root))
from src.analysis import (
    assign_prps,
    # apply_affinity_level_fixed
)

benchmark_root = repo_root / "data" / "results" / "affinity_benchmark"
programme_order = ["IBR", "Stability", "DER", "CROF", "System Services", "Planning"]
level_order = ["Low", "Moderate", "High"]
color_discrete_map = {"Low": "#86b6ef", "Moderate": "#2a78d6", "High": "#104281"}
# level_order = ["Moderate", "High"]

In [2]:
assignments = []
for path in sorted(benchmark_root.glob("tpwrs_*/adjudicated_ra_affinities.csv")):
    year = int(path.parent.name.split("_")[1])

    assignments.append(assign_prps(pd.read_csv(path)).assign(Year=year))

assignments_df = pd.concat(assignments, ignore_index=True)

# assignments_df = assignments_df[assignments_df["Affinity_Level"]!="Low"]

In [3]:
total_counts =  (
    assignments_df[assignments_df["Affinity_Level"].isin(level_order)]
    .groupby(["Year", "Affinity_Level"], observed=True)["Abstract_Index"]
    .nunique()
    .rename("Publications")
    .reset_index()
)
total_counts.head()
fig = px.line(
    total_counts, x="Year", y="Publications", color="Affinity_Level",
    markers=True,
    category_orders={"Affinity_Level": level_order},
    color_discrete_map= color_discrete_map,
    title="Relevant TPWRS Publications by PRP and Year",
)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.update_traces(line_width=2.5, marker_size=7)
fig.update_xaxes(title=None, dtick=1, tickformat="d", showticklabels=True)
fig.update_yaxes(title=None)
fig.update_yaxes(title="Publications", col=1)
# fig.update_layout(height=700)
fig.show()

In [4]:
level_dtype = pd.CategoricalDtype(categories=level_order, ordered=True)  # Low < Moderate < High
assignments_df["Affinity_Level"] = assignments_df["Affinity_Level"].astype(level_dtype)

overall_level = (
    assignments_df
    .sort_values(
        ["Year", "Abstract_Index", "Affinity_Level", "Highest_Question_Affinity", "PRP_Name"],
        ascending=[True, True, False, False, True],
    )
    .drop_duplicates(["Year", "Abstract_Index"], keep="first")[
        ["Year", "Abstract_Index", "Affinity_Level", "PRP_Name"]
    ]
    .rename(columns={"PRP_Name": "Top_PRP_Name"})
    .reset_index(drop=True)
)

overall_counts = (
    overall_level
    .groupby(["Year", "Affinity_Level"], observed=True)
    .size()
    .rename("Publications")
    .reset_index()
)
overall_counts["Probability"] = (
    overall_counts["Publications"]
    / overall_counts.groupby("Year")["Publications"].transform("sum")*100
)

fig = px.bar(
    overall_counts,
    x="Year",
    y="Probability",
    color="Affinity_Level",
    barmode="stack",
    category_orders={"Affinity_Level": level_order},
    color_discrete_map=color_discrete_map,
    title="Relevant TPWRS Publications by PRP and Year",
)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.update_traces(marker_line_width=2, marker_line_color="#fcfcfb")
fig.update_xaxes(title=None, dtick=1, tickformat="d", showticklabels=True)
fig.update_yaxes(title="Share of Publications", ticksuffix="%")
fig.show()
overall_counts.groupby("Year")["Publications"].sum()

Year
2015    358
2016    531
2017    498
2018    703
2019    511
2020    483
2021    559
2022    451
2023    499
2024    602
2025    452
2026    358
Name: Publications, dtype: int64

In [5]:
overall_level_grouped = overall_level.assign(
    # Low has no meaningful top programme, so bucket it separately from real PRP picks
    Top_PRP_Name=overall_level["Top_PRP_Name"].where(overall_level["Affinity_Level"] != "Low", "Other"),
    # Affinity_Level=overall_level["Affinity_Level"].astype(str).where(overall_level["Affinity_Level"] == "Low", "Moderate+High")
)

overall_counts_by_prp = (
    overall_level_grouped
    .groupby(["Year", "Affinity_Level", "Top_PRP_Name"], observed=True)
    .size()
    .rename("Publications")
    .reset_index()
)
# fig1 = px.scatter(
#     x = "Year",
#     # y = "Publications",
#     # color = "Affinity_Level",
#     y = "Top_PRP_Name",
#     size = "Publications",
#     color = "Publications",
#     data_frame = overall_counts_by_prp
# )
# fig1.show()
facet_order = ["Other"] + programme_order
fig2 = px.line(
    overall_counts_by_prp,
    x="Year",
    y="Publications",
    color="Affinity_Level",
    facet_col="Top_PRP_Name",
    facet_col_wrap=4,
    facet_row_spacing=0.12,
    markers=True,
    category_orders={"Top_PRP_Name": facet_order, "Affinity_Level": level_order},
    color_discrete_map=color_discrete_map,
    title="Top-Matching PRP Publications by Affinity Level and Year",
)
fig2.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig2.update_traces(line_width=2.5, marker_size=7)
fig2.update_xaxes(title=None, dtick=1, tickformat="d", showticklabels=True)
fig2.update_yaxes(title=None)
fig2.update_yaxes(title="Publications", col=1)
fig2.update_layout(height=700)
fig2.show()

overall_counts_by_prp.groupby("Year")["Publications"].sum()

Year
2015    358
2016    531
2017    498
2018    703
2019    511
2020    483
2021    559
2022    451
2023    499
2024    602
2025    452
2026    358
Name: Publications, dtype: int64

In [6]:
publication_counts = (
    assignments_df[assignments_df["Affinity_Level"].isin(level_order)]
    .groupby(["Year", "PRP_Name", "Affinity_Level"], observed=True)["Abstract_Index"]
    .nunique()
    .rename("Publications")
    .reset_index()
)

fig = px.line(
    publication_counts, x="Year", y="Publications", color="Affinity_Level",
    facet_col="PRP_Name", facet_col_wrap=3, facet_row_spacing=0.12, markers=True,
    category_orders={"PRP_Name": programme_order, "Affinity_Level": level_order},
    color_discrete_map=color_discrete_map,
    title="Relevant TPWRS Publications by PRP and Year",
)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.update_traces(line_width=2.5, marker_size=7)
fig.update_xaxes(title=None, dtick=1, tickformat="d", showticklabels=True)
fig.update_yaxes(title=None)
fig.update_yaxes(title="Publications", col=1)
fig.update_layout(height=700)
fig.show()
display(publication_counts.sort_values(["PRP_Name", "Year", "Affinity_Level"]))

,Year,PRP_Name,Affinity_Level,Publications
0,2015,CROF,Low,159
1,2015,CROF,Moderate,182
2,2015,CROF,High,17
18,2016,CROF,Low,207
19,2016,CROF,Moderate,299
...,...,...,...,...
196,2025,System Services,Moderate,120
197,2025,System Services,High,19
213,2026,System Services,Low,230
214,2026,System Services,Moderate,105


In [7]:
publications_per_year = (
    assignments_df.groupby("Year")["Abstract_Index"]
    .nunique()
    .rename("Total_Publications")
)

relative_publication_counts = publication_counts.merge(
    publications_per_year, on="Year", validate="many_to_one"
)
relative_publication_counts["Publication_Share"] = (
    100 * relative_publication_counts["Publications"]
    / relative_publication_counts["Total_Publications"]
)

relative_fig = px.line(
    relative_publication_counts, x="Year", y="Publication_Share", color="Affinity_Level",
    facet_col="PRP_Name", facet_col_wrap=3, facet_row_spacing=0.12, markers=True,
    category_orders={"PRP_Name": programme_order, "Affinity_Level": level_order},
    color_discrete_map=color_discrete_map,
    title="Share of TPWRS Publications Relevant to Each PRP by Year",
)
relative_fig.for_each_annotation(
    lambda annotation: annotation.update(text=annotation.text.split("=")[-1])
)
relative_fig.update_traces(line_width=2.5, marker_size=7)
relative_fig.update_xaxes(title=None, dtick=1, tickformat="d", showticklabels=True)
relative_fig.update_yaxes(title=None, ticksuffix="%", rangemode="tozero", dtick=10)
relative_fig.update_yaxes(title="Share of publications", col=1)
relative_fig.update_layout(height=700)
relative_fig.show()
display(
    relative_publication_counts.sort_values(["PRP_Name", "Year", "Affinity_Level"])
)

,Year,PRP_Name,Affinity_Level,Publications,Total_Publications,Publication_Share
0,2015,CROF,Low,159,358,44.413408
1,2015,CROF,Moderate,182,358,50.837989
2,2015,CROF,High,17,358,4.748603
18,2016,CROF,Low,207,531,38.983051
19,2016,CROF,Moderate,299,531,56.308851
...,...,...,...,...,...,...
196,2025,System Services,Moderate,120,452,26.548673
197,2025,System Services,High,19,452,4.203540
213,2026,System Services,Low,230,358,64.245810
214,2026,System Services,Moderate,105,358,29.329609


In [8]:
output_dir = repo_root / "notebooks" / "outputs" / "tpwrs_prp_publications_by_year"
output_dir.mkdir(parents=True, exist_ok=True)

relative_fig.write_image(
    str(output_dir / "tpwrs_prp_publications_share_by_year.png"),
    width=1400,
    height=700,
    scale=2
)
print(f"Figure saved to {output_dir / 'tpwrs_prp_publications_share_by_year.png'}")

Figure saved to /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/notebooks/outputs/tpwrs_prp_publications_by_year/tpwrs_prp_publications_share_by_year.png


In [9]:
save_to = r'/mnt/c/Users/pgallego/OneDrive - Imperial College London/RA/Research agenda/G-PST shared folder/IEEE-GPST Article/Shared documents/Abstracts classified'
if False:
    assignments_df.to_csv(f"{save_to}/assignments.csv", index=False)
    relative_publication_counts.to_csv(f"{save_to}/relative_publication_counts.csv", index=False)

In [10]:
publication_counts

,Year,PRP_Name,Affinity_Level,Publications
0,2015,CROF,Low,159
1,2015,CROF,Moderate,182
2,2015,CROF,High,17
3,2015,DER,Low,273
4,2015,DER,Moderate,81
...,...,...,...,...
211,2026,Stability,Moderate,176
212,2026,Stability,High,56
213,2026,System Services,Low,230
214,2026,System Services,Moderate,105


In [11]:
relative_publication_counts

,Year,PRP_Name,Affinity_Level,Publications,Total_Publications,Publication_Share
0,2015,CROF,Low,159,358,44.413408
1,2015,CROF,Moderate,182,358,50.837989
2,2015,CROF,High,17,358,4.748603
3,2015,DER,Low,273,358,76.256983
4,2015,DER,Moderate,81,358,22.625698
...,...,...,...,...,...,...
211,2026,Stability,Moderate,176,358,49.162011
212,2026,Stability,High,56,358,15.642458
213,2026,System Services,Low,230,358,64.245810
214,2026,System Services,Moderate,105,358,29.329609


In [12]:
assignments_df

,Abstract_Index,PRP_Name,Highest_Question_ID,Highest_Question_Text,Highest_Question_Affinity,Affinity_Level,Year
0,10.1109/TPWRS.2014.2316271,Stability,9,what analytical tools and models should be pro...,61.4,Moderate,2015
1,10.1109/TPWRS.2014.2316271,Planning,41,what additional probabilistic planning methods...,53.6,Moderate,2015
2,10.1109/TPWRS.2014.2316271,CROF,27,how can risk-based methods (e.g. probabilistic...,31.4,Low,2015
3,10.1109/TPWRS.2014.2316271,System Services,35,"what models and methods, including testing, ar...",19.8,Low,2015
4,10.1109/TPWRS.2014.2316271,DER,12,what mechanisms and appropriate aggregate der ...,15.8,Low,2015
...,...,...,...,...,...,...,...
36025,10.1109/TPWRS.2026.3707412,IBR,1,what is the future of active power balance as ...,75.8,Moderate,2026
36026,10.1109/TPWRS.2026.3707412,Stability,9,what analytical tools and models should be pro...,75.6,Moderate,2026
36027,10.1109/TPWRS.2026.3707412,System Services,36,what system services can be designed to enable...,45.0,Moderate,2026
36028,10.1109/TPWRS.2026.3707412,Planning,37,what studies and metrics are required to ident...,25.6,Low,2026
